# Rnadom Forest Classifier

In [10]:
import pandas as pd
import numpy as np
import time
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold, cross_validate, GridSearchCV
import itertools
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    make_scorer
)

## Data Preparation

In [4]:
# train test splits
def train_test_split(full_df):
  nrows = full_df.shape[0]

  train_size = int(0.8 * nrows)

  train_df = full_df[:train_size]
  test_df = full_df[train_size:]

  return train_df, test_df

# train-val split for Houldout Validation
def train_val_split(train_df):
  nrows = train_df.shape[0]

  train_size = int(0.9 * nrows)

  holdout_train_df = train_df[:train_size]
  holdout_val_df = train_df[train_size:]

  return holdout_train_df, holdout_val_df


# path to datasets
obesity_df_path = "obesity_shuffled_notscaled.csv"
depression_df_path = "depression_shuffled_notscaled.csv"
congressional_df_train_path = "congressional_df_train_preprocessed.csv"
congresional_df_test_path = "congressional_df_test_preprocessed.csv"
rev_df_train_path = "amazon_review_ID.shuf.lrn.csv"
rev_df_test_path = "amazon_review_ID.shuf.tes.csv"

# load datasets
obesity_df = pd.read_csv(obesity_df_path)
depression_df = pd.read_csv(depression_df_path)

# train-test sets
obesity_df_train, obesity_df_test = train_test_split(obesity_df)
depression_df_train, depression_df_test = train_test_split(depression_df)
congressional_df_train = pd.read_csv(congressional_df_train_path)
congressional_df_test = pd.read_csv(congresional_df_test_path)
rev_df_train = pd.read_csv(rev_df_train_path)
rev_df_train.drop(columns=['ID'], inplace=True)
rev_df_test = pd.read_csv(rev_df_test_path)

# train-val sets for holdout validation
obesity_df_train_holdout, obesity_df_val_holdout = train_val_split(obesity_df_train)
depression_df_train_holdout, depression_df_val_holdout = train_val_split(depression_df_train)
congressional_df_train_holdout, congressional_df_val_holdout = train_val_split(congressional_df_train)
rev_df_train_holdout, rev_df_val_holdout = train_val_split(rev_df_train)

## Function Definitions

In [7]:
def select_tree_hyperparameters_oob(df, target_attribute, main_scorer='f1'):

    param_grid = {
        'criterion': ["entropy", "gini"],
        'min_samples_split': range(2, 10, 2),
        'min_samples_leaf': range(2, 5, 1),
        'n_estimators': range(100, 601, 100)
    }

    # adaptive max_depth
    n_rows = df.shape[0]
    if n_rows <= 1000:
        param_grid["max_depth"] = range(4, 8, 2)
    elif n_rows <= 10000:
        param_grid["max_depth"] = range(6, 15, 2)
    else:
        param_grid["max_depth"] = range(10, 24, 2)

    x = df.loc[:, df.columns != target_attribute]
    y_raw = df[target_attribute]
    le = LabelEncoder()
    y = le.fit_transform(y_raw)

    results = []
    best_score = -np.inf
    best_model = None

    # Grid over tree params including n_estimators
    for criterion, min_split, min_leaf, max_depth, n_est in itertools.product(
        param_grid['criterion'],
        param_grid['min_samples_split'],
        param_grid['min_samples_leaf'],
        param_grid['max_depth'],
        param_grid['n_estimators']
    ):
        model = RandomForestClassifier(
            n_estimators=n_est,
            max_features='sqrt',
            bootstrap=True,
            oob_score=True,
            random_state=1,
            n_jobs=-1,
            criterion=criterion,
            min_samples_split=min_split,
            min_samples_leaf=min_leaf,
            max_depth=max_depth
        )
        model.fit(x, y)
        score = model.oob_score_

        # OOB predictions
        y_pred_oob = model.predict(x)
        acc = accuracy_score(y, y_pred_oob)
        prec = precision_score(y, y_pred_oob, average='macro', zero_division=0)
        rec = recall_score(y, y_pred_oob, average='macro', zero_division=0)
        f1 = f1_score(y, y_pred_oob, average='macro', zero_division=0)

        results.append({
            "criterion": criterion,
            "min_samples_split": min_split,
            "min_samples_leaf": min_leaf,
            "max_depth": max_depth,
            "n_estimators": n_est,
            "oob_score": score,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1
        })

        # track best model
        if score > best_score:
            best_score = score
            best_model = model
            best_params = {
                "criterion": criterion,
                "min_samples_split": min_split,
                "min_samples_leaf": min_leaf,
                "max_depth": max_depth,
                "n_estimators": n_est,
                "oob_score": score,
                "accuracy": acc,
                "precision": prec,
                "recall": rec,
                "f1": f1
            }

    # convert to DataFrame and sort
    results_df = pd.DataFrame(results).sort_values(by="oob_score", ascending=False).reset_index(drop=True)
    print("Top 5 OOB-based configs:")
    display(results_df.head(5))

    # Return best params, label encoder, results_df, and best trained model
    return best_params, le, results_df, best_model

# --- for rfc training with Grid Search and Cross Validation ---
# --- for rfc training with Grid Search and Cross Validation ---
def train_random_forest_with_grid_cv(df, target_attribute, main_scorer, no_estimators):

  scoring = {
    'accuracy': 'accuracy',
    'precision': make_scorer(precision_score, average='macro', zero_division=0),
    'recall': make_scorer(recall_score, average='macro', zero_division=0),
    'f1': make_scorer(f1_score, average='macro', zero_division=0),
  }

  param_grid = {
      'criterion': ["entropy", "gini"],
      # 'max_depth': [],  # defined below; acc to all datasets
      'min_samples_split': range(2,10,2),
      'min_samples_leaf': range(2,5,1),
  }

  if df.shape[0] <= 1000:
    param_grid["max_depth"] = range(4,8,2)
  elif df.shape[0] > 1000 and df.shape[0] <= 10000:
    param_grid["max_depth"] = range(6,15,2)
  else:
    param_grid["max_depth"] = range(10,24,2)

  # # decision process for max_depth
  # eg:
  #   in congressional data:
  #     218 samples
  #   max_depth => sample_sixe/(2^max_depth); contains [5-10]samples per leaf
  #   so here: 218/(2^5) = 6.8 samples per leaf

  #  => max-depth = log_base2(samples) - c[1,4]
  # param_grid['max_depth'] = math.log2(df.shape[0]).astype(int) - 2

  x = df.loc[:, df.columns != target_attribute]
  y_raw = df[target_attribute]
  le = LabelEncoder()
  y = le.fit_transform(y_raw)

  start = time.perf_counter()

  rfc = RandomForestClassifier(n_estimators=no_estimators,
                               max_features='sqrt', # default
                               bootstrap=True,
                               oob_score=True,
                               random_state=1, # for reprocucibility sake
                               n_jobs=-1 # how may processors to use in parallel
                               )

  cv_strategy = KFold(
    n_splits=5,
    shuffle=False
  )

  grid_search = GridSearchCV(
      estimator=rfc,
      param_grid=param_grid,
      cv=cv_strategy,
      scoring=scoring,
      refit=main_scorer,
      verbose=True
  )

  grid_search.fit(x,y)

  results = pd.DataFrame(grid_search.cv_results_)
  elapsed = time.perf_counter() - start
  results["Completion_time"] = elapsed
  print("best accuracy", grid_search.best_score_)
  print(grid_search.best_estimator_)
  print("Time(s): ", elapsed)
  return results, le, grid_search.best_estimator_


# ---  for rfc training with Grid Search and Holdout Validation ---
def train_random_forest_with_grid_holdout(df_train, df_val, target_attribute, main_scorer, no_estimators):

  scoring = {
    'accuracy': accuracy_score,
    'precision': lambda y_true, y_pred: precision_score(y_true, y_pred, average='macro'),
    'recall': lambda y_true, y_pred: recall_score(y_true, y_pred, average='macro'),
    'f1': lambda y_true, y_pred: f1_score(y_true, y_pred, average='macro')
  }

  param_grid = {
      'criterion': ["entropy", "gini"],
      # 'max_depth': [],  # defined below; acc to all datasets
      'min_samples_split': range(2,10,2),
      'min_samples_leaf': range(1,5,1),
      'max_features': ['sqrt', 'log2']
  }

  if df_train.shape[0] <= 1000:
    param_grid["max_depth"] = range(3,8,1)
  elif df_train.shape[0] <= 10000:
    param_grid["max_depth"] = range(5,15,1)
  else:
    param_grid["max_depth"] = range(10,25,1)

  x_train = df_train.loc[:, df_train.columns != target_attribute]
  y_train_raw = df_train[target_attribute]
  le = LabelEncoder()
  y_train = le.fit_transform(y_train_raw)

  x_val = df_val.loc[:, df_val.columns != target_attribute]
  y_val_raw = df_val[target_attribute]
  le = LabelEncoder()
  y_val = le.fit_transform(y_val_raw)

  x_full = pd.concat([x_train, x_val], axis=0)
  y_full = np.concatenate([y_train, y_val])

  start = time.perf_counter()

  param_combinations = list(itertools.product(
    param_grid['criterion'],
    param_grid['min_samples_split'],
    param_grid['min_samples_leaf'],
    param_grid['max_features'],
    param_grid['max_depth']
))

  best_score = 0
  best_model = None
  results_list = []

  for criterion, min_samples_split, min_samples_leaf, max_features, max_depth in param_combinations:
      model = RandomForestClassifier(
          n_estimators=no_estimators,
          bootstrap=True,
          oob_score=True,
          random_state=1,
          n_jobs=-1,
          criterion=criterion,
          min_samples_split=min_samples_split,
          min_samples_leaf=min_samples_leaf,
          max_features=max_features,
          max_depth=max_depth
      )
      model.fit(x_train, y_train)
      y_pred = model.predict(x_val)

      result = {
          'param_criterion': criterion,
          'param_min_samples_split': min_samples_split,
          'param_max_depth': max_depth,
          'accuracy': accuracy_score(y_val, y_pred),
          'precision': precision_score(y_val, y_pred, average='macro'),
          'recall': recall_score(y_val, y_pred, average='macro'),
          'f1': f1_score(y_val, y_pred, average='macro')
      }

      results_list.append(result)

      if result[main_scorer] > best_score:
          best_score = result[main_scorer]
          best_model = model

  final_model = RandomForestClassifier(
      criterion=best_model.get_params()['criterion'],
      min_samples_split=best_model.get_params()['min_samples_split'],
      min_samples_leaf=best_model.get_params()['min_samples_leaf'],
      max_features=best_model.get_params()['max_features'],
      max_depth=best_model.get_params()['max_depth'],
      random_state=1
  )

  final_model.fit(x_full, y_full)


  elapsed = time.perf_counter() - start
  results = pd.DataFrame(results_list)
  results['Completion_time'] = elapsed

  print("Best", main_scorer, best_score)
  print("Time(s):", elapsed)
  return results, le, final_model


# --- for getting top 5 results ---
def get_top_results_cv(results_df, main_scorer):
  mean_score_metrics = ["mean_test_f1", "mean_test_accuracy", "mean_test_precision", "mean_test_recall"]
  if f"mean_test_{main_scorer}" in mean_score_metrics:
    mean_score_metrics.remove(f"mean_test_{main_scorer}")
  mean_score_metrics.insert(0, f"mean_test_{main_scorer}")
  results_df["combined_rank"] = results_df["rank_test_accuracy"] + results_df["rank_test_precision"] + results_df["rank_test_recall"] + results_df["rank_test_f1"]
  results_df_sorted = results_df.sort_values(by=mean_score_metrics, ascending=False)
  param_cols = [col for col in results_df_sorted.columns if 'param_' in col]
  mean_score_metrics.append("combined_rank")
  relevant_cols = mean_score_metrics + param_cols
  results_df_sorted_relevant = results_df_sorted[relevant_cols]

  return results_df_sorted_relevant

def get_top_results_holdout(results_df, main_scorer):
  mean_score_metrics = ["f1", "accuracy", "precision", "recall"]
  if main_scorer in mean_score_metrics:
    mean_score_metrics.remove(main_scorer)
  mean_score_metrics.insert(0, main_scorer)
  results_df_sorted = results_df.sort_values(by=mean_score_metrics, ascending=False)
  param_cols = [col for col in results_df_sorted.columns if 'param_' in col]
  relevant_cols = mean_score_metrics + param_cols
  results_df_sorted_relevant = results_df_sorted[relevant_cols]

  return results_df_sorted_relevant


# --- for Prediction on Test Data ---
def pred_test_data(test_df, model, label_encoder, target_attribute, has_ground_truth):
  x_test = test_df.loc[:, test_df.columns != target_attribute]
  x_test_ids = []
  conf_matrix_df = pd.dataframe()
  if "ID" in x_test.columns:
    x_test_ids = x_test["ID"]
    x_test = x_test.drop(columns=["ID"])

  if has_ground_truth:
    y_test_raw = test_df[target_attribute]
    y_test = label_encoder.transform(y_test_raw)


  start = time.perf_counter()

  y_pred = model.predict(x_test)

  elapsed = time.perf_counter() - start
  final_results = pd.DataFrame()
  final_results["time"] = [elapsed]
  final_results["parameters"] = [model.get_params()]
  if has_ground_truth:
    final_results["accuracy"] = [accuracy_score(y_test, y_pred)]
    final_results["precision"] = [precision_score(y_test, y_pred, average="macro")]
    final_results["recall"] = [recall_score(y_test, y_pred, average="macro")]
    final_results["f1"] = [f1_score(y_test, y_pred, average="macro")]
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_labels = label_encoder.classes_
    conf_matrix_df = pd.DataFrame(conf_matrix, index=class_labels, columns=class_labels)
    conf_matrix_df.index.name = 'Actual'
    conf_matrix_df.columns.name = 'Predicted'
  final_pred = x_test.copy()
  final_pred["y_pred"] = label_encoder.inverse_transform(y_pred)
  if len(x_test_ids) > 0:
    final_pred["id"] = x_test_ids

  return final_results, final_pred, conf_matrix_df


## just

In [ ]:
# --- for rfc training with Grid Search and Cross Validation ---
def train_random_forest_with_grid_cv(df, target_attribute, main_scorer):

  scoring = {
    'accuracy': 'accuracy',
    'precision': make_scorer(precision_score, average='macro', zero_division=0),
    'recall': make_scorer(recall_score, average='macro', zero_division=0),
    'f1': make_scorer(f1_score, average='macro', zero_division=0),
  }

  param_grid = {
      'criterion': ["entropy", "gini"],
      # 'max_depth': [],  # defined below; acc to all datasets
      'min_samples_split': range(2,10,2),
      'min_samples_leaf': range(2,5,1),
  }

  if df.shape[0] <= 1000:
    param_grid["max_depth"] = range(4,8,2)
  elif df.shape[0] > 1000 and df.shape[0] <= 10000:
    param_grid["max_depth"] = range(6,15,2)
  else:
    param_grid["max_depth"] = range(10,24,2)

  # # decision process for max_depth
  # eg:
  #   in congressional data:
  #     218 samples
  #   max_depth => sample_sixe/(2^max_depth); contains [5-10]samples per leaf
  #   so here: 218/(2^5) = 6.8 samples per leaf

  #  => max-depth = log_base2(samples) - c[1,4]
  # param_grid['max_depth'] = math.log2(df.shape[0]).astype(int) - 2

  x = df.loc[:, df.columns != target_attribute]
  y_raw = df[target_attribute]
  le = LabelEncoder()
  y = le.fit_transform(y_raw)

  start = time.perf_counter()

  rfc = RandomForestClassifier(n_estimators=500, # jus any no bw 500-1000 coz diff datasets may overfit/underfit
                               max_features='sqrt', # default
                               bootstrap=True,
                               oob_score=True,
                               random_state=1, # for reprocucibility sake
                               n_jobs=-1 # how may processors to use in parallel
                               )

  cv_strategy = KFold(
    n_splits=5,
    shuffle=True,
    random_state=1
  )

  grid_search = GridSearchCV(
      estimator=rfc,
      param_grid=param_grid,
      cv=cv_strategy,
      scoring=scoring,
      refit=main_scorer,
      verbose=True
  )

  grid_search.fit(x,y)

  results = pd.DataFrame(grid_search.cv_results_)
  elapsed = time.perf_counter() - start
  results["Completion_time"] = elapsed
  print("best accuracy", grid_search.best_score_)
  print(grid_search.best_estimator_)
  print("Time(s): ", elapsed)
  return results, le, grid_search.best_estimator_




# ---  for rfc training with Grid Search and Holdout Validation ---
def train_random_forest_with_grid_holdout(df_train, df_val, target_attribute, main_scorer):

  param_grid = {
      'criterion': ["entropy", "gini"],
      # 'max_depth': [],  # defined below; acc to all datasets
      'min_samples_split': range(2,10,2),
      'min_samples_leaf': range(2,5,1),
  }

  if df_train.shape[0] <= 1000:
    param_grid["max_depth"] = range(4,8,2)
  elif df_train.shape[0] > 1000 and df_train.shape[0] <= 10000:
    param_grid["max_depth"] = range(6,15,2)
  else:
    param_grid["max_depth"] = range(10,24,2)

  x_train = df_train.loc[:, df_train.columns != target_attribute]
  y_train_raw = df_train[target_attribute]
  le = LabelEncoder()
  le.fit(pd.concat([df_train[target_attribute], df_val[target_attribute]]))
  y_train = le.transform(y_train_raw)

  x_val = df_val.loc[:, df_val.columns != target_attribute]
  y_val_raw = df_val[target_attribute]
  y_val = le.transform(y_val_raw)

  x_full = pd.concat([x_train, x_val], axis=0)
  y_full = np.concatenate([y_train, y_val])

  start = time.perf_counter()

  param_combinations = list(itertools.product(
    param_grid['criterion'],
    param_grid['min_samples_split'],
    param_grid['min_samples_leaf'],
    param_grid['max_depth']
))

  best_score = 0
  best_model = None
  results_list = []

  for criterion, min_samples_split, min_samples_leaf, max_depth in param_combinations:
      model = RandomForestClassifier(
          n_estimators=500,
          max_features='sqrt',
          bootstrap=True,
          oob_score=True,
          random_state=1,
          n_jobs=-1,
          criterion=criterion,
          min_samples_split=min_samples_split,
          min_samples_leaf=min_samples_leaf,
          max_depth=max_depth,
      )
      model.fit(x_train, y_train)
      y_pred = model.predict(x_val)

      result = {
          'param_criterion': criterion,
          'param_min_samples_split': min_samples_split,
          'param_max_depth': max_depth,
          'accuracy': accuracy_score(y_val, y_pred),
          'precision': precision_score(y_val, y_pred, average='macro', zero_division=0),
          'recall': recall_score(y_val, y_pred, average='macro', zero_division=0),
          'f1': f1_score(y_val, y_pred, average='macro', zero_division=0),
      }

      results_list.append(result)

      if result[main_scorer] > best_score:
          best_score = result[main_scorer]
          best_model = model

  final_model = RandomForestClassifier(
      criterion=best_model.get_params()['criterion'],
      min_samples_split=best_model.get_params()['min_samples_split'],
      min_samples_leaf=best_model.get_params()['min_samples_leaf'],
      max_features=best_model.get_params()['max_features'],
      max_depth=best_model.get_params()['max_depth'],
      random_state=1
  )

  final_model.fit(x_full, y_full)


  elapsed = time.perf_counter() - start
  results = pd.DataFrame(results_list)
  results['Completion_time'] = elapsed

  print("Best", main_scorer, best_score)
  print("Time(s):", elapsed)
  return results, le, final_model


# --- for getting top 5 results ---
def get_top_results_cv(results_df, main_scorer):
  mean_score_metrics = ["mean_test_f1", "mean_test_accuracy", "mean_test_precision", "mean_test_recall"]
  if f"mean_test_{main_scorer}" in mean_score_metrics:
    mean_score_metrics.remove(f"mean_test_{main_scorer}")
  mean_score_metrics.insert(0, f"mean_test_{main_scorer}")
  results_df["combined_rank"] = results_df["rank_test_accuracy"] + results_df["rank_test_precision"] + results_df["rank_test_recall"] + results_df["rank_test_f1"]
  results_df_sorted = results_df.sort_values(by=mean_score_metrics, ascending=False)
  param_cols = [col for col in results_df_sorted.columns if 'param_' in col]
  mean_score_metrics.append("combined_rank")
  relevant_cols = mean_score_metrics + param_cols
  results_df_sorted_relevant = results_df_sorted[relevant_cols]

  return results_df_sorted_relevant

def get_top_results_holdout(results_df, main_scorer):
  mean_score_metrics = ["f1", "accuracy", "precision", "recall"]
  if main_scorer in mean_score_metrics:
    mean_score_metrics.remove(main_scorer)
  mean_score_metrics.insert(0, main_scorer)
  results_df_sorted = results_df.sort_values(by=mean_score_metrics, ascending=False)
  param_cols = [col for col in results_df_sorted.columns if 'param_' in col]
  relevant_cols = mean_score_metrics + param_cols
  results_df_sorted_relevant = results_df_sorted[relevant_cols]

  return results_df_sorted_relevant


# --- for Prediction on Test Data ---
def pred_test_data(test_df, model, label_encoder, target_attribute, has_ground_truth):
  x_test = test_df.loc[:, test_df.columns != target_attribute]
  x_test_ids = []
  if "ID" in x_test.columns:
    x_test_ids = x_test["ID"]
    x_test = x_test.drop(columns=["ID"])

  if has_ground_truth:
    y_test_raw = test_df[target_attribute]
    y_test = label_encoder.transform(y_test_raw)


  start = time.perf_counter()

  y_pred = model.predict(x_test)

  elapsed = time.perf_counter() - start
  final_results = pd.DataFrame()
  final_results["time"] = [elapsed]
  final_results["parameters"] = [model.get_params()]
  if has_ground_truth:
    final_results["accuracy"] = [accuracy_score(y_test, y_pred)]
    final_results["precision"] = [precision_score(y_test, y_pred, average="macro")]
    final_results["recall"] = [recall_score(y_test, y_pred, average="macro")]
    final_results["f1"] = [f1_score(y_test, y_pred, average="macro")]
  final_pred = x_test.copy()
  final_pred["y_pred"] = label_encoder.inverse_transform(y_pred)
  if len(x_test_ids) > 0:
    final_pred["id"] = x_test_ids

  return final_results, final_pred


## OOB Scores validation with Grid Search

In [134]:
# for obesity
obesity_best_params_oob, obesity_le_oob, obesity_oob_results, obesity_best_model_oob = select_tree_hyperparameters_oob(
    df=obesity_df_train, 
    target_attribute="obesity_level_grouped", 
    main_scorer="accuracy")

Top 5 OOB-based configs:


,criterion,min_samples_split,min_samples_leaf,max_depth,n_estimators,oob_score,accuracy,precision,recall,f1
0,entropy,6,2,12,500,0.959123,0.996445,0.994345,0.994488,0.994388
1,entropy,6,2,14,400,0.959123,0.997038,0.995353,0.995624,0.995473
2,entropy,6,2,12,400,0.958531,0.996445,0.994345,0.994488,0.994388
3,entropy,4,2,12,400,0.957938,0.997038,0.995353,0.995624,0.995473
4,entropy,2,4,12,200,0.957938,0.986967,0.981188,0.981741,0.981188


In [135]:
# for depression dataset
depression_best_params_oob, depression_le_oob, depression_oob_results, depression_best_model_oob = select_tree_hyperparameters_oob(
    df=depression_df_train, 
    target_attribute="depression", 
    main_scorer="recall")

Top 5 OOB-based configs:


,criterion,min_samples_split,min_samples_leaf,max_depth,n_estimators,oob_score,accuracy,precision,recall,f1
0,gini,8,3,18,200,0.844333,0.892082,0.893447,0.884096,0.887890
1,gini,4,2,20,400,0.844064,0.917728,0.919533,0.911154,0.914675
2,gini,2,2,20,400,0.844064,0.917728,0.919533,0.911154,0.914675
3,gini,4,2,22,600,0.843974,0.924588,0.926395,0.918466,0.921837
4,gini,2,2,22,600,0.843974,0.924588,0.926395,0.918466,0.921837


In [ ]:
# Select tree-specific hyperparameters (using OOB or CV)
congressional_best_params_oob, congressional_le_oob, congressional_oob_results, congressional_best_model_oob = select_tree_hyperparameters_oob(
    df=congressional_df_train, 
    target_attribute="class", 
    main_scorer="f1")

Top 5 OOB-based configs:


,criterion,min_samples_split,min_samples_leaf,max_depth,n_estimators,oob_score,accuracy,precision,recall,f1
0,entropy,8,2,4,100,0.958716,0.967890,0.966356,0.967768,0.967040
1,entropy,8,3,4,100,0.958716,0.967890,0.966356,0.967768,0.967040
2,entropy,2,2,4,100,0.954128,0.977064,0.975759,0.977200,0.976457
3,entropy,4,3,4,600,0.954128,0.967890,0.966356,0.967768,0.967040
4,gini,6,3,4,500,0.954128,0.967890,0.966356,0.967768,0.967040


In [136]:
# for amazon reviews
rev_best_params_oob, rev_le_oob, rev_oob_results, rev_best_model_oob = select_tree_hyperparameters_oob(
    df=rev_df_train, 
    target_attribute="Class", 
    main_scorer="f1")

KeyboardInterrupt: 

## Cross Validation with Grid Search

### Training and Hyperparameter Tuning

In [ ]:
results_obesity_cv, le_obesity_cv, obesity_best_model_cv, conf_matrix_obesity_df = train_random_forest_with_grid_cv(obesity_df_train, "obesity_level_grouped", "accuracy", 500)

Fitting 5 folds for each of 120 candidates, totalling 600 fits


In [12]:
results_depression_cv, le_depression_cv, depression_best_model_cv = train_random_forest_with_grid_cv(depression_df_train, "depression", "recall", 200)

Fitting 5 folds for each of 168 candidates, totalling 840 fits
best accuracy 0.8365159583695382
RandomForestClassifier(max_depth=22, min_samples_leaf=2, n_estimators=200,
                       n_jobs=-1, oob_score=True, random_state=1)
Time(s):  820.5537747992203


In [13]:
results_congressional_cv, le_congressional_cv, congressional_best_model_cv = train_random_forest_with_grid_cv(congressional_df_train, "class", "accuracy", 100)

Fitting 5 folds for each of 48 candidates, totalling 240 fits
best accuracy 0.9542283298097252
RandomForestClassifier(criterion='entropy', max_depth=4, min_samples_leaf=2,
                       min_samples_split=6, n_jobs=-1, oob_score=True,
                       random_state=1)
Time(s):  47.3946079723537


In [14]:
results_rev_cv, le_rev_cv, rev_best_model_cv = train_random_forest_with_grid_cv(rev_df_train, "Class", "f1", 200)

Fitting 5 folds for each of 48 candidates, totalling 240 fits
best accuracy 0.3951083464022808
RandomForestClassifier(max_depth=6, min_samples_leaf=4, n_estimators=200,
                       n_jobs=-1, oob_score=True, random_state=1)
Time(s):  356.44247905816883


### Top Results

In [15]:
processed_results_obesity_cv = get_top_results_cv(results_obesity_cv, "accuracy")
display('Top Results for Obesity',processed_results_obesity_cv.head())

'Top Results for Obesity'

,mean_test_accuracy,mean_test_f1,mean_test_precision,mean_test_recall,combined_rank,param_criterion,param_max_depth,param_min_samples_leaf,param_min_samples_split
48,0.956160,0.942060,0.943687,0.943119,4,entropy,14,2,2
49,0.956160,0.942060,0.943687,0.943119,4,entropy,14,2,4
50,0.956160,0.941617,0.943527,0.942162,10,entropy,14,2,6
36,0.954975,0.940224,0.942426,0.940756,16,entropy,12,2,2
37,0.954975,0.940224,0.942426,0.940756,16,entropy,12,2,4


In [16]:
processed_results_depression_cv = get_top_results_cv(results_depression_cv, "recall")
display('Top Results for Depression',processed_results_depression_cv.head())

'Top Results for Depression'

,mean_test_recall,mean_test_f1,mean_test_accuracy,mean_test_precision,combined_rank,param_criterion,param_max_depth,param_min_samples_leaf,param_min_samples_split
156,0.836516,0.839501,0.845364,0.843945,4,gini,22,2,2
157,0.836516,0.839501,0.845364,0.843945,4,gini,22,2,4
159,0.835139,0.838195,0.844154,0.842795,19,gini,22,2,8
63,0.835028,0.838290,0.844378,0.843298,13,entropy,20,2,8
74,0.834993,0.838045,0.844019,0.842630,37,entropy,22,2,6


In [17]:
processed_results_congressional_cv = get_top_results_cv(results_congressional_cv, "f1")
display('Top Results for Congressional Voting',processed_results_congressional_cv.head())

'Top Results for Congressional Voting'

,mean_test_f1,mean_test_accuracy,mean_test_precision,mean_test_recall,combined_rank,param_criterion,param_max_depth,param_min_samples_leaf,param_min_samples_split
19,0.952666,0.954123,0.94908,0.958551,23,entropy,6,3,8
20,0.952666,0.954123,0.94908,0.958551,23,entropy,6,4,2
21,0.952666,0.954123,0.94908,0.958551,23,entropy,6,4,4
22,0.952666,0.954123,0.94908,0.958551,23,entropy,6,4,6
23,0.952666,0.954123,0.94908,0.958551,23,entropy,6,4,8


In [18]:
processed_results_rev_cv = get_top_results_cv(results_rev_cv, "f1")
display('Top Results for Amazon Reviews',processed_results_rev_cv.head())

'Top Results for Amazon Reviews'

,mean_test_f1,mean_test_accuracy,mean_test_precision,mean_test_recall,combined_rank,param_criterion,param_max_depth,param_min_samples_leaf,param_min_samples_split
44,0.395108,0.446667,0.430696,0.480044,22,gini,6,4,2
45,0.395108,0.446667,0.430696,0.480044,22,gini,6,4,4
46,0.395108,0.446667,0.430696,0.480044,22,gini,6,4,6
47,0.395108,0.446667,0.430696,0.480044,22,gini,6,4,8
38,0.394161,0.437333,0.438527,0.470504,35,gini,6,2,6


### Predictions on Test Data

In [19]:
prediction_results_obesity_cv, y_pred_obesity_cv = pred_test_data(obesity_df_test, obesity_best_model_cv, le_obesity_cv, "obesity_level_grouped", True)
display('Obesity',prediction_results_obesity_cv)

'Obesity'

,time,parameters,accuracy,precision,recall,f1
0,0.123143,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.973995,0.962778,0.967956,0.965298


In [20]:
prediction_results_depression_cv, y_pred_depression_cv = pred_test_data(depression_df_test, depression_best_model_cv, le_depression_cv, "depression", True)
display('Depression',prediction_results_depression_cv)

'Depression'

,time,parameters,accuracy,precision,recall,f1
0,0.082592,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.848458,0.844846,0.836843,0.840262


In [21]:
prediction_results_congressional_cv, y_pred_congressional_cv = pred_test_data(congressional_df_test, congressional_best_model_cv, le_congressional_cv, "class", False)
display('Congressional Voting',prediction_results_congressional_cv)

'Congressional Voting'

,time,parameters
0,0.031176,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w..."


In [22]:
prediction_results_rev_cv, y_pred_rev_cv = pred_test_data(rev_df_test, rev_best_model_cv, le_rev_cv, "class", False)
display('Amazon Reviews:',prediction_results_rev_cv)

'Amazon Reviews:'

,time,parameters
0,0.121456,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w..."


## Holdout with Grid Search

### Training and Hyperparameter Tuning

In [137]:
results_obesity_holdout, le_obesity_holdout, obesity_best_model_holdout = train_random_forest_with_grid_holdout(obesity_df_train_holdout, obesity_df_val_holdout, "obesity_level_grouped", "accuracy")

Best accuracy 0.9408284023668639
Time(s): 523.9321648750047


In [138]:
results_depression_holdout, le_depression_holdout, depression_best_model_holdout = train_random_forest_with_grid_holdout(depression_df_train_holdout, depression_df_val_holdout, "depression", "recall")

KeyboardInterrupt: 

In [139]:
results_congressional_holdout, le_congressional_holdout, congressional_best_model_holdout = train_random_forest_with_grid_holdout(congressional_df_train_holdout, congressional_df_val_holdout, "class", "f1")

Best f1 1.0
Time(s): 181.80166433400882


In [140]:
results_rev_holdout, le_rev_holdout, rev_best_model_holdout = train_random_forest_with_grid_holdout(rev_df_train_holdout, rev_df_val_holdout, "Class", "f1")

/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetri

Best f1 0.03623188405797101
Time(s): 1560.3838172919932


### Top Results

In [ ]:
processed_results_obesity_holdout = get_top_results_holdout(results_obesity_holdout, "accuracy")
display('Obesity',processed_results_obesity_holdout.head())

In [ ]:
processed_results_depression_holdout = get_top_results_holdout(results_depression_holdout, "recall")
display('Depression',processed_results_depression_holdout.head())

In [ ]:
processed_results_congressional_holdout = get_top_results_holdout(results_congressional_holdout, "f1")
display('Congressional Voting Holdout results',processed_results_congressional_holdout.head())

'Congressional Voting Holdout results'

,accuracy,f1,precision,recall,param_criterion,param_min_samples_split,param_max_depth
0,1.0,1.0,1.0,1.0,entropy,2,3
5,1.0,1.0,1.0,1.0,entropy,2,3
10,1.0,1.0,1.0,1.0,entropy,2,3
11,1.0,1.0,1.0,1.0,entropy,2,4
15,1.0,1.0,1.0,1.0,entropy,2,3


In [ ]:
processed_results_rev_holdout = get_top_results_holdout(results_rev_holdout, "f1")
display('Amazon Reviews',processed_results_rev_holdout.head())

### Predictions on Test Data

In [ ]:
prediction_results_obesity_holdout, y_pred_obesity_holdout = pred_test_data(obesity_df_test, obesity_best_model_holdout, le_obesity_holdout, "obesity_level_grouped", True)
display('Obesity',prediction_results_obesity_holdout.head())

In [ ]:
prediction_results_depression_holdout, y_pred_depression_holdout = pred_test_data(depression_df_test, depression_best_model_holdout, le_depression_holdout, "depression", True)
display('Depression',prediction_results_depression_holdout.head())

In [41]:
prediction_results_congressional_holdout, y_pred_congressional_holdout = pred_test_data(congressional_df_test, congressional_best_model_holdout, le_congressional_holdout, "class", False)
display('Congressional Voting',prediction_results_congressional_holdout.head())

'Congressional Voting'

,time,parameters
0,0.083271,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w..."


In [ ]:
prediction_results_rev_holdout, y_pred_rev_holdout = pred_test_data(rev_df_test, rev_best_model_holdout, le_rev_holdout, "class", False)
display('Amazon Reviews',prediction_results_rev_holdout.head())

## Prediction files preparation for Kaggle

In [25]:
def kaggle_comp_file(pred_df):
  pred_df_final = pred_df[["id", "y_pred"]].rename(columns={"y_pred": "class", "id" : "ID"})
  return pred_df_final

In [26]:
# Congressional Voting

#kaggle_submission_congressional_oob = kaggle_comp_file(y_pred_congressional_oob)
kaggle_submission_congressional_cv = kaggle_comp_file(y_pred_congressional_cv)
# kaggle_submission_congressional_holdout = kaggle_comp_file(y_pred_congressional_holdout)

#kaggle_submission_congressional_oob.to_csv('congressional_random_forest_cv_submission_group39.csv', index=False)
kaggle_submission_congressional_cv.to_csv('congressional_random_forest_cv_submission_group39.csv', index=False)
# kaggle_submission_congressional_holdout.to_csv('kaggle_preds/congressional_random_forest_holdout_submission_group39.csv', index=False)

In [27]:
# Amazon Reviews
kaggle_submission_rev_cv = kaggle_comp_file(y_pred_rev_cv)
#kaggle_submission_rev_holdout = kaggle_comp_file(y_pred_rev_holdout)

kaggle_submission_rev_cv.to_csv('reviews_random_forest_cv_submission_group39.csv', index=False)
#kaggle_submission_rev_holdout.to_csv('kaggle_preds/reviews_random_forest_holdout_submission_group39.csv', index=False)    

In [40]:
# y_pred_congressional_cv.head()